In [1]:
import os, subprocess

def load_amber_env():
    """Load AmberTools + Amber environment into the Jupyter kernel."""
    bash_cmd = """
    module purge
    module load StdEnvACCRE/2023
    module load gcc/12.3 openmpi/4.1.5 cuda/12.2
    module load ambertools/25.0
    module load scipy-stack/2025a
    module load amber/24
    env
    """
    env_output = subprocess.check_output(["bash", "-c", bash_cmd], text=True)
    for line in env_output.splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            os.environ[k] = v
    print("✅ Amber environment loaded into Jupyter kernel.")


In [2]:
load_amber_env()

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) CCconfig        6)  ucx/1.14.1         11) flexiblas/3.3.1
  2) gentoo/2023     7)  libfabric/1.18.0   12) blis/0.9.0
  3) gcccore/.12.3   8)  pmix/4.2.4         13) StdEnvACCRE/2023
  4) gcc/12.3        9)  ucc/1.2.0
  5) hwloc/2.9.1     10) openmpi/4.1.5
Lmod Warning:
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: scipy-stack/2023b
(required by: plumed/2.9.0, ambertools/25.0)
-------------------------------------------------------------------------------




The following have been reloaded with a version change:
  1) ipykernel/2023b => ipykernel/2025a
  2) scipy-stack/2023b => scipy-stack/2025a



✅ Amber environment loaded into Jupyter kernel.


Lmod Warning:
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: scipy-stack/2025a
(required by: amber/24)
-------------------------------------------------------------------------------




Inactive Modules:
  1) arrayfire/3.9.0     2) cudnn/8.9.5.29     3) glfw/3.3.8     4) spdlog/1.11.0

Due to MODULEPATH changes, the following have been reloaded:
  1) ambertools/25.0     2) plumed/2.9.0     3) ucc-cuda/1.2.0     4) ucx-cuda/1.14.1

The following have been reloaded with a version change:
  1) cuda/12.2 => cuda/12.6
  2) cudacore/.12.2.2 => cudacore/.12.6.2
  3) ipykernel/2025a => ipykernel/2023b
  4) nccl/2.18.3 => nccl/2.26.2
  5) scipy-stack/2025a => scipy-stack/2023b



## 🧬 **Cα Publish Pipeline — Summary**

For each system, the workflow:

- 🔍 Reads the three `combined*_nowat_ions.nc` trajectories with the matching stripped topology.  
- ✂️ Strips all atoms except `:1–275@CA` and writes a correct **Cα-only topology**.  
- 🔗 Generates three Cα-only replicate trajectories, concatenates them into **`Ca_combined.nc`**, and deletes the intermediate files.  
- 🔒 Leaves the original data untouched and writes only **`Ca_combined.prmtop`** and **`Ca_combined.nc`** into the publish directory.


In [3]:
%%bash
set -euo pipefail

SRC_BASE="/panfs/accrepfs.vampire/data/kojetin_lab/julian/combined_files"
OUT_BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"
mkdir -p "$OUT_BASE"

systems=(
  3PRG
  6ONI_apo
  6ONI_pep_nolig
  6ONI_ligand
  6ONI_pep_lig
  6ONJ_apo
  6ONJ_pep_nolig
  6ONJ_ligand
  6ONJ_pep_lig
)

echo "🧬 Publishing CA-only trajectories into:"
echo "    $OUT_BASE"
echo

for sys in "${systems[@]}"; do
  SRC="${SRC_BASE}/${sys}"
  OUT="${OUT_BASE}/${sys}"
  mkdir -p "$OUT"

  echo "➡️  ${sys}"

  # -----------------------------------------
  # Stage 1: generate CA-only reps + CA.prmtop
  #   - IMPORTANT: use strip ... parmout logic
  # -----------------------------------------
  cpptraj <<EOF
parm ${SRC}/prepinHMR_stripped.prmtop

# Rep 1 (also writes CA topology via parmout)
trajin ${SRC}/combined1_nowat_ions.nc
autoimage
strip !(:1-275@CA) parmout ${OUT}/Ca.prmtop
trajout ${OUT}/Ca_rep1.nc netcdf
run
clear trajin
clear trajout

# Rep 2
trajin ${SRC}/combined2_nowat_ions.nc
autoimage
strip !(:1-275@CA)
trajout ${OUT}/Ca_rep2.nc netcdf
run
clear trajin
clear trajout

# Rep 3
trajin ${SRC}/combined3_nowat_ions.nc
autoimage
strip !(:1-275@CA)
trajout ${OUT}/Ca_rep3.nc netcdf
run
quit
EOF

  # -----------------------------------------
  # Stage 2: concatenate using the CA.prmtop
  # -----------------------------------------
  cpptraj <<EOF
parm ${OUT}/Ca.prmtop
trajin ${OUT}/Ca_rep1.nc
trajin ${OUT}/Ca_rep2.nc
trajin ${OUT}/Ca_rep3.nc
trajout ${OUT}/Ca_combined.nc netcdf
run
quit
EOF

  # -----------------------------------------
  # Stage 3: cleanup intermediates (output only)
  # -----------------------------------------
  rm -f ${OUT}/Ca_rep1.nc ${OUT}/Ca_rep2.nc ${OUT}/Ca_rep3.nc

  # Optional: keep naming consistent with your earlier convention
  mv -f ${OUT}/Ca.prmtop ${OUT}/Ca_combined.prmtop

  echo "   ✅ ${OUT}/Ca_combined.prmtop"
  echo "   ✅ ${OUT}/Ca_combined.nc"
  echo
done

echo "🎯 Done: all systems CA-stripped + concatenated (input untouched)."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Publishing CA-only trajectories into:
    /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025

➡️  3PRG

CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 01/13/26 09:05:32
| Available memory: 25.226 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/julian/combined_files/3PRG/prepinHMR_stripped.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/julian/combined_files/3PRG/prepinHMR_stripped.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/julian/combined_files/3PRG/combined1_nowat_ions.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/julian/combined_files/3PRG/combined1_nowat_ions.nc' as Amber NetCDF
  [autoimage]
    AUTOIMAGE: To box center based on center of mass, anchor is first molecule.
	Auto-imaging using molecule dist

## 🧬 **Master Cα Concatenation — Round 2**

This step generates only two master trajectories in  
`Publish_cMD_12_27_2025/`:

- **6ONI_ALL_Ca_combined.nc** — concatenation of all 6ONI systems  
- **6ONJ_ALL_Ca_combined.nc** — concatenation of all 6ONJ systems  

The **apo `Ca_combined.prmtop`** is used as the reference topology for both.

⚠️ **3PRG is not concatenated** because it exists only as a single apo system — its `Ca_combined.nc` inside the `3PRG/` folder is already final.


In [4]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"

echo "🧬 Creating master Cα trajectories in:"
echo "    $BASE"
echo

# -------------------------
# 6ONI — concatenate all
# -------------------------
cpptraj <<EOF
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_apo/Ca_combined.nc
trajin ${BASE}/6ONI_pep_nolig/Ca_combined.nc
trajin ${BASE}/6ONI_ligand/Ca_combined.nc
trajin ${BASE}/6ONI_pep_lig/Ca_combined.nc
trajout ${BASE}/6ONI_ALL_Ca_combined.nc netcdf
run
quit
EOF
echo "✅ 6ONI_ALL_Ca_combined.nc created"

# -------------------------
# 6ONJ — concatenate all
# -------------------------
cpptraj <<EOF
parm ${BASE}/6ONJ_apo/Ca_combined.prmtop
trajin ${BASE}/6ONJ_apo/Ca_combined.nc
trajin ${BASE}/6ONJ_pep_nolig/Ca_combined.nc
trajin ${BASE}/6ONJ_ligand/Ca_combined.nc
trajin ${BASE}/6ONJ_pep_lig/Ca_combined.nc
trajout ${BASE}/6ONJ_ALL_Ca_combined.nc netcdf
run
quit
EOF
echo "✅ 6ONJ_ALL_Ca_combined.nc created"

echo
echo "🎯 Done — only 6ONI_ALL_Ca_combined.nc and 6ONJ_ALL_Ca_combined.nc written."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Creating master Cα trajectories in:
    /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 01/13/26 09:14:14
| Available memory: 10.420 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.nc' as Amber NetCDF
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_pep_nolig/Ca_combined.nc]
	Reading '

### Create global Cα master trajectory

Combine the Cα trajectories in the order **6ONI_ALL → 6ONJ_ALL → 3PRG** into a single NetCDF trajectory: `ALL_SYSTEMS_Ca_combined.nc`.

In [6]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"

echo "🧬 Creating global Cα master trajectory:"
echo "    Order: 6ONI_ALL → 6ONJ_ALL → 3PRG"
echo

cpptraj <<EOF
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/6ONI_ALL_Ca_combined.nc
trajin ${BASE}/6ONJ_ALL_Ca_combined.nc
trajin ${BASE}/3PRG/Ca_combined.nc
trajout ${BASE}/ALL_SYSTEMS_Ca_combined.nc netcdf
run
quit
EOF

echo "🎯 ALL_SYSTEMS_Ca_combined.nc created successfully."


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🧬 Creating global Cα master trajectory:
    Order: 6ONI_ALL → 6ONJ_ALL → 3PRG


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools) OpenMP
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| 16 OpenMP threads available.
| Date/time: 01/13/26 09:16:37
| Available memory: 10.437 GB

INPUT: Reading input from 'STDIN'
  [parm /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.prmtop]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.prmtop' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_ALL_Ca_combined.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_ALL_Ca_combined.nc' as Amber NetCDF
  [trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONJ_ALL_Ca_combined.nc]
	Reading '/panfs/accrepfs.vampire/data/kojetin_lab/

# 🧠 Job 1 — PCA for each individual system

In [ ]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"
SLURM_DIR="${BASE}/Slurm_Scripts"
mkdir -p "$SLURM_DIR"
cd "$SLURM_DIR"

cat > PCA_group1.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PCA_group1
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --gres=gpu:nvidia_titan_x:1
#SBATCH --time=5-00:00:00
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=PCA_group1_out_%j.log
#SBATCH --error=PCA_group1_err_%j.log

module --force purge
module load StdEnvACCRE/2023
module load ambertools/25.0
module load amber/24
export OMP_NUM_THREADS=1

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"

systems=(3PRG 6ONI_apo 6ONI_pep_nolig 6ONI_ligand 6ONI_pep_lig 6ONJ_apo 6ONJ_pep_nolig 6ONJ_ligand 6ONJ_pep_lig)

for sys in "${systems[@]}"; do
  PCADIR="${BASE}/${sys}/PCA"
  mkdir -p "$PCADIR"
  cd "$PCADIR"

  cpptraj <<EOT
parm ../Ca_combined.prmtop
trajin ../Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name Modes
crdaction trajectory projection modes Modes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT
done

EOF

sbatch PCA_group1.slurm


# 🧠 Job 2 — PCA for 6ONI_ALL and 6ONJ_ALL

In [ ]:
%%bash
cd "/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/Slurm_Scripts"

cat > PCA_group2.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PCA_group2
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --gres=gpu:nvidia_titan_x:1
#SBATCH --time=5-00:00:00
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=PCA_group2_out_%j.log
#SBATCH --error=PCA_group2_err_%j.log

module --force purge
module load StdEnvACCRE/2023
module load ambertools/25.0
module load amber/24
export OMP_NUM_THREADS=1

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"

for SYS in 6ONI 6ONJ; do
  PCADIR="${BASE}/${SYS}_ALL_PCA"
  mkdir -p "$PCADIR"
  cd "$PCADIR"

  cpptraj <<EOT
parm ${BASE}/${SYS}_apo/Ca_combined.prmtop
trajin ${BASE}/${SYS}_ALL_Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name Modes
crdaction trajectory projection modes Modes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT
done

EOF

sbatch PCA_group2.slurm


# 🧠 Job 3 — PCA for ALL_SYSTEMS_Ca_combined

In [7]:
%%bash
cd "/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/Slurm_Scripts"

cat > PCA_group3.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PCA_group3
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --gres=gpu:nvidia_titan_x:1
#SBATCH --time=5-00:00:00
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=PCA_group3_out_%j.log
#SBATCH --error=PCA_group3_err_%j.log

module --force purge
module load StdEnvACCRE/2023
module load ambertools/25.0
module load amber/24
export OMP_NUM_THREADS=1

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"
cd "$BASE"

cpptraj <<EOT
parm ${BASE}/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/ALL_SYSTEMS_Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name AllModes
crdaction trajectory projection modes AllModes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT

EOF

sbatch PCA_group3.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 8400938
